In [1]:
import os
from pathlib import Path

# Set this before importing jax.
JAX_CPU_ONLY = True
if JAX_CPU_ONLY:
    os.environ["JAX_PLATFORMS"] = "cpu"
    os.environ["CUDA_VISIBLE_DEVICES"] = ""

from jax import random
from omegaconf import OmegaConf

from dpraco.data import get_data_stream
from dpraco.models import get_model
from dpraco.training.training_routine import train_and_evaluate

import dpraco
import jax

# After editing the library: Kernel > Restart Session, then Run All.
# Confirm this notebook is using the repo checkout you expect:
print("dpraco:", dpraco.__file__)
print("JAX devices:", jax.devices())

dpraco: /home/iocebere/dpraco/src/dpraco/__init__.py
JAX devices: [CpuDevice(id=0)]


In [2]:
def resolve_repo_root() -> Path:
    """Prefer the directory that contains ``conf/dpraco.yaml`` (``examples/`` or repo root)."""
    cwd = Path.cwd().resolve()
    if (cwd / "conf" / "dpraco.yaml").is_file():
        return cwd
    if (cwd / "examples" / "conf" / "dpraco.yaml").is_file():
        return cwd / "examples"
    return cwd


EXP_FOLDER = resolve_repo_root()
CONF = EXP_FOLDER / "conf"
DATA_ROOT = EXP_FOLDER.parent / "data"


def load_cfg():
    base_cfg = OmegaConf.load(CONF / "dpraco.yaml")
    dataset_cfg = OmegaConf.load(CONF / "dataset" / "celeba.yaml")
    algorithm_cfg = OmegaConf.load(CONF / "algorithm" / "dpraco_demographic_parity.yaml")
    model_cfg = OmegaConf.load(CONF / "model" / "resnet16.yaml")

    dataset_cfg.data_path = DATA_ROOT / "celeba_3splits"

    cfg = OmegaConf.merge(
        base_cfg,
        OmegaConf.create(
            {
                "dataset": dataset_cfg,
                "algorithm": algorithm_cfg,
                "model": model_cfg,
            }
        ),
    )

    cfg.training_params.seed = 2189
    cfg.training_params.target_epsilon = 1.0
    cfg.training_params.number_of_steps = 1
    cfg.training_params.batch_size = 64
    cfg.training_params.eval_batch_size = 64
    cfg.eval.eval_every_k = 1
    # Hard cap on eval batches per split (train/val/test). Without this, CPU runs can look
    # "stuck" for a long time due to XLA compile + many batches.
    cfg.eval.max_batches_per_dataset = 2

    # Small splits so TFDS iteration stays cheap (still capped by max_batches above).
    cfg.dataset.max_train_samples = 256
    cfg.dataset.max_val_samples = 128
    cfg.dataset.max_test_samples = 128

    return cfg


def run_training(cfg):
    import sys
    import time

    def _log(msg: str) -> None:
        print(msg, flush=True, file=sys.stderr)

    t0 = time.perf_counter()

    key = random.PRNGKey(cfg.training_params.seed)
    data_key, model_key, training_key = random.split(key, num=3)

    _log("[smoke] building data streams (TFDS may be slow the first time)...")
    train_stream, val_data, test_data = get_data_stream(
        cfg, data_key, seed=cfg.training_params.seed
    )
    _log(f"[smoke] data ready ({time.perf_counter() - t0:.1f}s); building model...")
    state = get_model(cfg, model_key)
    _log(
        "[smoke] starting train+eval. First JAX compile on CPU often takes several minutes; "
        "this is normal."
    )
    metrics_df = train_and_evaluate(
        cfg=cfg,
        state=state,
        train_stream=train_stream,
        rng=training_key,
        test_data=test_data,
        val_data=val_data,
    )
    _log(f"[smoke] finished in {time.perf_counter() - t0:.1f}s")
    return metrics_df

In [3]:
cfg = load_cfg()

{
    "dataset": cfg.dataset.name,
    "model": cfg.model.name,
    "algorithm": cfg.algorithm.name,
    "steps": cfg.training_params.number_of_steps,
    "batch_size": cfg.training_params.batch_size,
    "eval_batch_size": cfg.training_params.eval_batch_size,
    "data_path": str(cfg.dataset.data_path),
    "max_train_samples": cfg.dataset.max_train_samples,
}

{'dataset': 'celeba',
 'model': 'resnet16',
 'algorithm': 'dpraco',
 'steps': 1,
 'batch_size': 64,
 'eval_batch_size': 64,
 'data_path': '/home/iocebere/dpraco/data/celeba_3splits',
 'max_train_samples': 256}

In [4]:
metrics_df = run_training(cfg)
metrics_df.tail(1)

[smoke] building data streams (TFDS may be slow the first time)...
/home/iocebere/anaconda3/envs/fairdp_sgd/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[smoke] data ready (15.6s); building model...
[smoke] starting train+eval. First JAX compile on CPU often takes several minutes; this is normal.


[train] building loss, update rule, and shard_map jit...
[train] entering training loop (first batch next)...
[train] step 1: first gradient update (XLA compile; often minutes on CPU)...
[train] step 1: running eval on train/val/test...
[eval:train] batch 0 (n=64)
[eval:train] batch 1 (n=64)
[eval:test] batch 0 (n=64)
[eval:test] batch 1 (n=64)
[eval:val] batch 0 (n=64)
[eval:val] batch 1 (n=64)


[smoke] finished in 179.9s


,step,train_loss,regularizer_avg,train_accuracy,train_soft_constraint,train_hard_constraint,train_learning_loss,train_precision,train_recall,train_f1-score,...,val_loss,val_soft_constraint,val_hard_constraint,val_learning_loss,val_precision,val_recall,val_f1-score,val_support,gamma,algorithm
0,1,0.781112,0.0,0.46875,0.02124621,0.0,0.781112,0.46875,1.0,0.638298,...,0.751124,0.011071205,0.0,0.751124,0.5,1.0,0.666667,64.0,0.3,dpraco
